# 可选实验——简单神经网络
在本实验中，我们将使用 Numpy 构建一个小型神经网络。它与您在 TensorFlow 中实现的“咖啡烘焙”网络相同。
   <center> <img  src="./images/C2_W1_CoffeeRoasting.png" width="400" />   <center/>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
import tensorflow as tf
from lab_utils_common import dlc, sigmoid
from lab_coffee_utils import load_coffee_data, plt_roast, plt_prob, plt_layer, plt_network, plt_output_unit
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

## 数据集
这与上一个实验使用的是同一个数据集。

In [ ]:
X,Y = load_coffee_data();
print(X.shape, Y.shape)

下面绘制咖啡烘焙数据。两个特征分别是以摄氏度为单位的温度和以分钟为单位的时长。[在家烘焙咖啡](https://www.merchantsofgreencoffee.com/how-to-roast-green-coffee-in-your-oven/) 建议将时长保持在 12 到 15 分钟之间，同时将温度保持在 175 到 260 摄氏度之间。当然，温度越高，时长应越短。

In [ ]:
plt_roast(X,Y)

### 数据归一化
为与上一个实验保持一致，我们将对数据进行归一化。更多细节请参阅该实验。

In [ ]:
print(f"Temperature Max, Min pre normalization: {np.max(X[:,0]):0.2f}, {np.min(X[:,0]):0.2f}")
print(f"Duration    Max, Min pre normalization: {np.max(X[:,1]):0.2f}, {np.min(X[:,1]):0.2f}")
norm_l = tf.keras.layers.Normalization(axis=-1)
norm_l.adapt(X)  # learns mean, variance
Xn = norm_l(X)
print(f"Temperature Max, Min post normalization: {np.max(Xn[:,0]):0.2f}, {np.min(Xn[:,0]):0.2f}")
print(f"Duration    Max, Min post normalization: {np.max(Xn[:,1]):0.2f}, {np.min(Xn[:,1]):0.2f}")

## NumPy 模型（使用 NumPy 进行前向传播）
<center> <img  src="./images/C2_W1_RoastingNetwork.PNG" width="200" />   <center/>
让我们构建课程中介绍的“咖啡烘焙网络”。它包含两个使用 sigmoid 激活的层。

如课程中所述，可以使用 NumPy 构建自己的稠密层。随后可用它来构建多层神经网络。

<img src="images/C2_W1_dense2.PNG" width="600" height="450">

在第一个可选实验中，你分别在 NumPy 和 TensorFlow 中构建了一个神经元，并注意到了两者的相似性。一个层只是包含多个神经元/单元。如课程中所述，可以使用 for 循环访问该层中的每个单元（`j`），计算该单元的权重（`W[:,j]`）与输入的点积，再加上该单元的偏置（`b[j]`），从而得到 `z`。然后，可以对结果应用激活函数 `g(z)`。下面让我们尝试构建一个“稠密层”子程序。

In [ ]:
def my_dense(a_in, W, b, g):
    """
    Computes dense layer
    Args:
      a_in (ndarray (n, )) : Data, 1 example 
      W    (ndarray (n,j)) : Weight matrix, n features per unit, j units
      b    (ndarray (j, )) : bias vector, j units  
      g    activation function (e.g. sigmoid, relu..)
    Returns
      a_out (ndarray (j,))  : j units|
    """
    units = W.shape[1]
    a_out = np.zeros(units)
    for j in range(units):               
        w = W[:,j]                                    
        z = np.dot(w, a_in) + b[j]         
        a_out[j] = g(z)               
    return(a_out)

下面的单元格使用上述 `my_dense` 子例程构建一个三层神经网络。

In [ ]:
def my_sequential(x, W1, b1, W2, b2):
    a1 = my_dense(x,  W1, b1, sigmoid)
    a2 = my_dense(a1, W2, b2, sigmoid)
    return(a2)

我们可以从上一个 TensorFlow 实验中复制训练好的权重和偏置。

In [ ]:
W1_tmp = np.array( [[-8.93,  0.29, 12.9 ], [-0.1,  -7.32, 10.81]] )
b1_tmp = np.array( [-9.82, -9.28,  0.96] )
W2_tmp = np.array( [[-31.18], [-27.59], [-32.56]] )
b2_tmp = np.array( [15.41] )

### 预测
<img align="left" src="./images/C2_W1_RoastingDecision.PNG"     style=" width:380px; padding: 10px 20px; " >

模型训练完成后，就可以用它进行预测。回想一下，模型的输出是一个概率，在本例中即烘焙效果良好的概率。要做出决策，必须将概率与阈值进行比较。在本例中，我们将使用 0.5

首先编写一个类似 TensorFlow `model.predict()` 的例程。它接收矩阵 $X$，其中各行包含全部 $m$ 个样本，并通过运行模型进行预测。

In [ ]:
def my_predict(X, W1, b1, W2, b2):
    m = X.shape[0]
    p = np.zeros((m,1))
    for i in range(m):
        p[i,0] = my_sequential(X[i], W1, b1, W2, b2)
    return(p)

我们可以在两个样本上试用此例程：

In [ ]:
X_tst = np.array([
    [200,13.9],  # postive example
    [200,17]])   # negative example
X_tstn = norm_l(X_tst)  # remember to normalize
predictions = my_predict(X_tstn, W1_tmp, b1_tmp, W2_tmp, b2_tmp)

为了将概率转换为决策，我们应用一个阈值：

In [ ]:
yhat = np.zeros_like(predictions)
for i in range(len(predictions)):
    if predictions[i] >= 0.5:
        yhat[i] = 1
    else:
        yhat[i] = 0
print(f"decisions = \n{yhat}")

可以用更简洁的方式完成：

In [ ]:
yhat = (predictions >= 0.5).astype(int)
print(f"decisions = \n{yhat}")

## 网络函数

该图展示了整个网络的运行过程，与上一个实验中的 TensorFlow 结果完全相同。
左图是最后一层的原始输出，以蓝色阴影表示，并叠加在由 X 和 O 表示的训练数据上。  
右图是应用决策阈值后的网络输出。这里的 X 和 O 对应网络作出的决策。  

In [ ]:
netf= lambda x : my_predict(norm_l(x),W1_tmp, b1_tmp, W2_tmp, b2_tmp)
plt_network(X,Y,netf)

## 恭喜！
你已经在 NumPy 中构建了一个小型神经网络。
希望本实验揭示了构成神经网络层的函数其实相当简单而且十分熟悉。